***
# AirBnB Listings in Zurich: Data analysis
***
Before delving into the analysis of Airbnb listings in the city of Zurich, let's install all necessary libraries:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import geopandas as gpd
import folium
import re
import umap

In [ ]:
import os
os.makedirs("images", exist_ok=True)

In [ ]:
%load_ext autoreload
%autoreload 2

***
## 1. About the Project
### 1.1. AirBnB listings as a topic
The short-term rental market, AirBnB in particular, has grown rapidly in urban areas, influencing rents, local economies, and urban planning. In cities like Zürich, understanding the factors of Airbnb prices can provide insights into market dynamics, potential regulatory interventions, and correlations with traditional rental prices. 
### 1.2. The Datasets
Inside AirBnB, the provider of our main dataset, is a non-commercial third-party provider of AirBnB data that aims to increase transparency of Airbnb activity. They scrape the official Airbnb website regularly and structure it csv files which can be obtained via their website https://insideairbnb.com/.
In addition, we are using two other datasets, one for normalizing the airbnb listings over neighborhoods in Zurich and the other to compare these normalized listing quantities with rental prices in each neighborhood.
### 1.3. Our Goals
In our project we aim to analyse Airbnb listings in Zürich, identify key features influencing pricing, and compare them with municipal rental statistics.
***
## 2. Loading, Cleaning and Validating the datasets
### 2.1. Loading and displaying dataframe overviews


In [ ]:
# define an overview function
def overview(df, name="dataset"):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True)
    }).sort_values(["dtype", "missing_pct"], ascending=[True, False])

    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    return summary

#loading and displaying
## Airbnb
listings_path = 'data/listings.csv'
airbnb_df= pd.read_csv(listings_path, encoding="utf-8")
print(overview(airbnb_df, "Airbnb"))

## housing stock
housing_path = 'data/bau522od5221_wohnungsbestand_zurich.csv'
housing_df = pd.read_csv(housing_path, encoding="utf-8")
display(overview(housing_df, "Housing"))

## Rental prices
rental_path = 'data/rental_prices.csv'
rental_df = pd.read_csv(rental_path, encoding="utf-8")
display(overview(rental_df,"Rental"))

### 2.2. Cleaning
Based on the overview, we have decided to clean the datasets the following way:
- Generally:
    - Standardize column names (lowercase, underscores etc.).
    - Turn every column with 2 unique values to boolean.
    - All object data types are converted to more primitive dtypes (string, float, int, category, sorted category).
- Airbnb:
    - Drop unnecessary columns (i.e. everything with a URL).
    - Remove columns with too much unusable noise, like *host_description* in *airbnb_df*. Instead, we added a column that states the length of the entry.
    - Reference all dates or temporal variables to the date the data was scraped.
    - Feature engineering amenities in the Airbnb dataframe by extracting amenities from the list in the *amenities* column.
- Housing Stock:
    - Only keep entries from 2025
    - Restructure the dataframe to only have 32 rows(one for each neighborhood). Aggregate the number of objects into separate columns for different apartment sizes.
- Rental Prices:
    - Only keep entries for 2024 (remove 2022)
    - Only keep square meter entries
    - Only keep netto entries (to remove differences in accounting etc. between neighborhoods as much as possible)
    - Restructure the dataframe to only have 32 rows (one for each neighborhood). The columns now represent different rental prices.

A unique normalization function was defined for each of the 3 datasets in the file *normalize.py*. Consult this file for further detail on the normalization process.

In [ ]:
from src.normalize import normalize_airbnb, normalize_housing, normalize_rental

# normalize Airbnb df
airbnb_df_norm = normalize_airbnb(airbnb_df)

# normalize housing stock df
housing_df_norm = normalize_housing(housing_df, year=2025)

# normalize rental price df
rental_df_norm = normalize_rental(rental_df, year=2024, brutto=False, sqm=True, level = 5, cat_zimmer=False)

***
## 3. Aggregating to quartier-level
At this point, the housing stock dataframe and rental price dataframe contain hundreds of rows, where each Quartier has several entries. For our analysis on quartier-level, we prefer a different structure, where each quartier only has one row. For that reson, we create a new pivot table around the quartier-column.
### 3.1. Using Pivot Tables

In [ ]:
from src.aggregation import quartiere_standardized_housing, quartiere_standardized_rental

# housing stock
housing_df_quart = quartiere_standardized_housing(housing_df_norm)
# flatten columns of housing df to 1:
housing_df_quart.columns = [
    "_".join(col).strip() if isinstance(col, tuple) else col
    for col in housing_df_quart.columns
]
housing_df_quart = housing_df_quart.drop(columns="index_")
display(housing_df_quart.head())

# rental prices
rental_df_quart = quartiere_standardized_rental(rental_df_norm)
display(rental_df_quart.head())

Now, the two datasets are of length 35 (i.e. both only contain one row for each of the 34 quartiere).
### 3.2. Add Airbnb count to Housing Stock dataframe
In order to compare airbnb listings (normalized by housing stock) and rental prices in each quartier, we need to know how many listings are in each quartier. Luckily, INside Airbnb provides the quartier that each listing lies in, so we simply need to count them and merge that count to *housing_df_quart*:

In [ ]:
# count airbnbs per quartier
counts_airbnb_quartier = (
    airbnb_df_norm
    .groupby("neighbourhood_cleansed")
    .size()
    .rename("n_airbnbs")
)

# merge
housing_df_quart = housing_df_quart.merge(
    counts_airbnb_quartier,
    left_on="quarlang_",
    right_index=True,
    how="left"
)

***
## 4. Exploratory Data Analysis
### 4.1. Airbnb Dataset
#### 4.1.1. Price
Let's simply calculate the minimum, maximum, mean, median and variance for `"price"` in the dataset:

In [ ]:
from src.functions import min_max_mean_med_var
min_max_mean_med_var(airbnb_df_norm["price"], "Airbnb prices")

Now, let us take a look at the distribution of listing prices in the dataset. We generate a histogram and a boxplot. Since there are some heavy outliers with very high prices, let us also crop the two plots to inspect the distribution at 'normal' price ranges (between 0 and 1000).

In [ ]:
from src.plots import plot_price_hist_box
plot_price_hist_box(airbnb_df_norm)

These plots show the following:
- **Positive skew**: mostly prices between 0 and 500, but outliers all the way at 10000
    - i.e. many cheap and moderately priced listings with few very expensive listings.
    - This means that we should be looking at the median rather than the mean when trying to get one metric for price.
- There is a **singular peak**: no submarkets visible (e.g. high demand for luxury listings could have resultet in a small peak at higher prices).
- **25th and 75th quantile** lie at roughly **100 CHF and 200 CHF** respectively.
#### 4.1.2. Reviews
There are 7 variables for different review scores. Since we aim to use some of them in our analysis later on, it would be helpful to see how their values are distributed.

In [ ]:
from src.plots import plot_multiple_violinplot
cols = ["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin", "review_scores_communication", "review_scores_location", "review_scores_value"]
col_names = ["Overall", "Accuracy", "Cleanliness", "Checkin", "Communication", "Location", "Value"]
plot_multiple_violinplot(airbnb_df, cols, col_names, 1.0, 5.0, title="Review score distributions", ylabel="Score (1–5)")

As expected, all review categories are mostly positive, with **medians around 4.7**. Of course, there are also **outliers toward the bottom**, as can be expected when asking for user feedback. We are especially **interested in the last variable: Value** since we will try to predict those values. We can see, that it has **by far the lowest mean** and one of the **largest variances**. We can tell by this shape, that it is heavily left-skewed. We will probably have to **transform it** in order to use it in a regression.
#### 4.1.3. Geographic distribution
Since each listing contains values for longitude and latitude, we can plot them on a map to get a sense of how they are distributed geographically across the city. We will add the borders of the quartiere aswell as colour the listings according to their corresponding quartier.  
For this process, we need to temporarily transform our airbnb listings dataframe into a geodataframe using geopandas:

In [ ]:
# convert airbnb_df_norm to geo data frame
airbnb_gdf = gpd.GeoDataFrame(
    airbnb_df_norm,
    geometry=gpd.points_from_xy(airbnb_df_norm["longitude"], airbnb_df_norm["latitude"])).set_crs(epsg=4326)

# import quartiere borders and forested areas
filepath_quart = "data/zurich_quartiere.gpkg"
filepath_for = "data/forest_zurich.gpkg"
layer_quart = "stzh.adm_statistische_quartiere_map"
quartiere = gpd.read_file(filepath_quart, layer=layer_quart).to_crs(epsg=4326)
forest = gpd.read_file(filepath_for).to_crs(epsg=4326)

# create map with function from src/maps.py
from src.maps import zurich_map_eda
zurich_map = zurich_map_eda(airbnb_gdf, forest, quartiere)
zurich_map

By **hovering over the empty land in the quartier**, the name of the quartier shows up. The color ramp uses the log-transform of the price, so the legend is rather vague. We mainly use the colors to convey the general distribution of listing prices. By hovering over a listing, one can see the actual price which should give an idea of how the color ramp works.
We can see that airbnb **listings are clustered along the lake shore and in the city center**. Forested areas obviously do not contain any listings, but listings reach the edge basically anywhere else.  
Concerning the **price**, we can see that prices are generally **higher in the city center and lower at the municipalitites borders**, although clusters of high prices can be found outside the center like on the lake shores or in Höngg. This price distribution will become important later on when we will try to train a model which will **predict listing prices based on a set of parameters**. Looking at this map, we guess that **location could play a major part in that model**.
#### 4.1.4. Correlation of dataset variables
Since we will be using several variables from the Airbnb dataset to predict things, it could be useful to see how these variables correlate with each other. All boolean variables have been converted back to 1s and 0s for this part. Then, we only took the numeric variables for the correlation matrix and dropped a bunch of variables we think do not fit for this visualization and would simply clutter the graph.  
Aigain, we are using a function for plotting the correlation matrix which is defined in our `src` folder.

In [ ]:
from src.plots import prepare_corr_df, plot_corr_heatmap

airbnb_corr = prepare_corr_df(airbnb_df_norm)
plot_corr_heatmap(airbnb_corr)

Take-aways from this correlation plot:
- obvious correlation between similar variables like `availability_eoy`and other `availability_xxx`variables. Or all the review scores among themselves.
- `host_is_superhost` has medium correlation with quite a lot of other variables (at least compared to most other pairings).
- Surprisingly to us, `price` does not show even moderate correlation with any variable except `estimated_revenue_l365d` (latter seems to be a function of the prior anyways). Knowing this, we expect it will be hard to find a set of variables which can be used to train a reliable model to predict listing price, however we can still consult this correlation matrix to get a starting set of variables, which we think might have the highest influence on the model.
- In general, negtive correlation between variables is much less common and less extreme than positive correlation. This means that there is no variable that causes another variable to decrease significantly if itself increased.

#### Plotting the correlation for different amenities results in the following:

In [ ]:
from src.plots import plot_amenity_price_corr
plot_amenity_price_corr(airbnb_df_norm)

- As can be seen there is a particularly high correlation to price and air conditioning and tv, followed by dishwasher and free parking
- The number of amenities stated in the listing also correlates positively with price
- Somewhat surprising, refrigirators and stoves negatively correlate with price, despite being meaningul for quality of life during the stay

### 4.2. Housing Stock Dataset
#### 4.2.1. Basic statistics of housing stock dataset
First, let us look at how the different stock-sizes (e.g. `_1-Zimmer`or `_5-Zimmer`) are distributed across the quartiere with boxplots:

In [ ]:
# restructure to get columns by number of rooms
from src.plots import restructure_housing_by_rooms, box_and_stacked_housing_stock
housing_rooms = restructure_housing_by_rooms(housing_df_quart)

# normalize for each quartier
housing_share = housing_rooms.div(housing_rooms.sum(axis=1), axis=0)
# plotting using src/plots.py
box_and_stacked_housing_stock(housing_share)

What we can gather from the boxplot:  
- Objects with 3 rooms are the most common. This follows typical housing structure patterns where the most common sizes are 3 or 4 rooms and they get less common towards the extremes like 1 room or 6 rooms.
- There is noticeable difference in variance (e.g. larger variance for 1 room than for 5 rooms), but these differences are not extreme.
- There are a couple of outliers which we can inspect in the stacked bar chart to the right.
- The two outliers from the boxplot for 1 room are now visible: Hochschulen and Rathaus have significantly more stock with 1 room than the others. This might be due to more student housing in this area.  
- The two outliers for 6 rooms are Hottingen and Fluntern, both quartiere lie on the slopes of the Zürichberg. It makes sense that one would find larger estates there, since they are considered more rich parts of the city.  
#### 4.2.2. Geographic distribution of airbnb stock
In order to get an idea of where the airbnb listings are situated, not on the level of each individual listing, but on the level of the quartiere, we can create a simple choropleth map. But first, we need to join the airbnb counts from `housing_df_quart` with the gpkg file containing the quartiere and their boundaries:
- import quartiere as gdf
- merge the number of airbnbs to the gdf
- compute airbnbs per n units of housing stock (we will also look at basic statistics of this variable first)
- plot the sorted gdf

In [ ]:
# calculate airbnb listings per n units of housing stock
n = 1000
housing_df_quart_density = housing_df_quart.copy()
housing_df_quart_density["airbnb_density"] = (housing_df_quart_density["n_airbnbs"] / housing_df_quart_density["total_units_"])*n

# set index
if housing_df_quart_density.index.name != "quarlang_":
    housing_df_quart_density = housing_df_quart_density.set_index("quarlang_")

# sort
housing_df_quart_density = housing_df_quart_density.sort_values("airbnb_density", ascending=False)

from src.functions import min_max_mean_med_var
min_max_mean_med_var(housing_df_quart_density["airbnb_density"], "Airbnbs per 1000 units of housing stock")

from src.plots import plot_per_quartier
plot_per_quartier(housing_df_quart_density, "airbnb_density", "Airbnb listings per 1000 units of housing stock", "Airbnb density", grid=True)

We can also plot this on a map. For that we need to import the quartiere geometry and merge the relevant columns to that `GeoDataFrame`. Then, we can use our generic plotting function for the quartiere of Zurich.

In [ ]:
# import the quartiere gpkg
filepath_quart = "data/zurich_quartiere.gpkg"
layer_quart = "stzh.adm_statistische_quartiere_map"
quartiere_gdf = gpd.read_file(filepath_quart, layer=layer_quart).to_crs(epsg=2056)

# perform join by attribute: quarlang_
quartiere_airbnb_gdf = quartiere.merge(
    housing_df_quart_density[["quarsort_", "n_airbnbs", "total_units_", "airbnb_density"]],
    left_on="qnr",
    right_on="quarsort_",
    how="left"
).to_crs(epsg=2056)

from src.maps import quartier_map
quartier_map(quartiere_airbnb_gdf, "airbnb_density", "Airbnb listings per 1000 units of housing stock")

The standout quartier is definetly *Hochschulen* with a value above 100 Airbnb listings per 1000 normal housing units. More precisely, this quartier has 354 total units of housing stock and 38 Airbnb lilstings. This ration far outscores even the runner up which is *Rathaus*.  

It is also evident from the map that the density of airbnb listings increases the closer we get to the city center (taken the center is somewhere around the main station).

### 4.3. Rental Prices Dataset
#### 4.3.1. Geographic distribution of rental prices
To get a broad overview of how rental prices differ accross the quartiere in Zurich, we just plot the mean rent for each quartier. But first, let us take a look at some basic statistics for this variable:

In [ ]:
# setting index to quartier name
if rental_df_quart.index.name != "gliederunglang":
    rental_df_quart = rental_df_quart.set_index("gliederunglang")

rental_df_quart = rental_df_quart.sort_values("rent_mean_both", ascending=False)

from src.functions import min_max_mean_med_var
min_max_mean_med_var(rental_df_quart["rent_mean_both"], "Mean rent")

# plot
plot_per_quartier(rental_df_quart, "rent_mean_both", "Rent in each quartier", "average rent per sqm [CHF]", grid=True)

There are some similarities in which quartiere are at the top of this ranking when comparing it to the previous graph. Though this correlation must be investigated further, since it could just be an artefact of these quartiere being closer to the city center than others (e.g. *Hochschulen*, *Rathaus*, *Werd*, *Sihlfeld*, *Langstrasse* etc.)  
There is also a significant difference in the variance, which will be explored in a boxplot later in the EDA process.  

To map this data, we first need to join the prices to the gdf that contains the quartiere boundaries:

In [ ]:
# load gpkg as gdf of quartiere boundaries
filepath_quart = "data/zurich_quartiere.gpkg"
layer_quart = "stzh.adm_statistische_quartiere_map"
quartiere_gdf = gpd.read_file(filepath_quart, layer=layer_quart).to_crs(epsg=2056)


# join rental prices by common attribute
quartiere_rent_gdf = quartiere.merge(
    rental_df_quart.set_index("gliederungsort")[["rent_mean_both"]],
    left_on="qnr",
    right_on="gliederungsort",
    how="left"
).to_crs(epsg=2056)

# map
from src.maps import quartier_map
quartier_map(quartiere_rent_gdf, "rent_mean_both", "Average rent per sqm in Zurich [CHF]")


- The highest rents are in the center of the city which is to be expected:
    - Lindenhof:	29.54
    - Rathaus:  	27.85
    - Hochschulen:	26.21
    - Hottingen:    25.85
    - Seefeld:  	25.34
- Some of these quartiere do also have high shares of airbnb listings, like Rathaus and Lindenhof.
### 4.4. Comparing rental prices and housing stock variance
As mentioned in step *4.3.1. Price for each quartier*, there is a difference in variance between rental prices and airbnb density across the quartiere in the city. We can show this by looking  at the following boxplot (for more detail, we also show the basic statistics again):

In [ ]:
# merge datasets for easier plot handling
rental_temp = rental_df_quart.copy().reset_index()
housing_temp = housing_df_quart_density.copy().reset_index()

temp_df = rental_temp[["rent_mean_both", "gliederunglang"]].merge(
    housing_temp[["quarlang_", "airbnb_density"]],
    left_on="gliederunglang",
    right_on="quarlang_",
    how="left"
)

from src.functions import min_max_mean_med_var
min_max_mean_med_var(temp_df["airbnb_density"], "Airbnbs per 1000 units of housing stock")
print("\n")
min_max_mean_med_var(temp_df["rent_mean_both"], "Mean rent")

cols = ["airbnb_density", "rent_mean_both"]
col_names = ["Airbnbs per 1000 units", "Mean rent"]
plot_multiple_boxplot(temp_df, cols, col_names, 0.0, 110.0, title="Variance comparison", grid=True)

We could already see the difference in variance when looking at the plots in steps *4.2.2. Geographic distribution of airbnb stock* and *4.3.1. Geographic distribution of rental prices*. Although the differing variances will not pose problems later on, when doing linear regression analysis with these two variables (actually, we will be using `"n_airbnbs"/"mean_rent_both"`. That changes the variable by a factor of 1000), we can already detect some potential problems:
- The Airbnb variable is skewed towards higher density.
- Mean rent also has a slight skew towards higher values.
- There is one outlier in the Aribnb variable which might be severely influential, so that must be taken into account once we get to evaluating the linear regression model's output.

This concludes the EDA of the three datasets. The following parts will focus on more in depth analysis and model training.

***
## 5. Clustering

### 5.1. Loading and Feature Selection

We derive two complementary feature sets from `airbnb_df_norm` (already normalized in §2.2):

- **Set A — "Quality profile"**: the 7 `review_scores_*` dimensions.  
  Answers: *"How do listings cluster by perceived service quality?"*
- **Set B — "Market segment"**: review scores + `price` (log-transformed) + `accommodates` + `amenities_count`.  
  Answers: *"How do listings cluster by overall market positioning?"*

> **Missing values**: `review_scores_*` contain NaN for listings with no reviews.  
> We apply `dropna()` rather than imputation — filling in median scores would inject false "neutral preference" signal.  
> The share of dropped rows is printed below.

In [ ]:
from src.clustering import build_feature_matrix, REVIEW_COLS, STRUCTURAL_COLS

X_quality, idx_quality = build_feature_matrix(airbnb_df_norm, set_name="quality")
X_market,  idx_market  = build_feature_matrix(airbnb_df_norm, set_name="market")

n_total    = len(airbnb_df_norm)
n_dropped_q = n_total - len(X_quality)
n_dropped_m = n_total - len(X_market)
print(f"Set A (quality): {X_quality.shape}  — dropped {n_dropped_q} rows ({n_dropped_q/n_total:.1%})")
print(f"Set B (market):  {X_market.shape}  — dropped {n_dropped_m} rows ({n_dropped_m/n_total:.1%})")

In [ ]:
display(X_quality.describe().T.round(2))
display(X_market.describe().T.round(2))

***
### 5.2. Standardization

Review scores span 1–5, `price` spans 0–10 000 CHF, `accommodates` ranges 1–16.  
The order-of-magnitude differences in scale would bias K-Means (distance-based) toward high-variance features.  
We apply `StandardScaler` to both feature sets before clustering or running PCA.

Note: `price` is already `log1p`-transformed inside `build_feature_matrix()`, consistent with the log-scale used in the EDA map (§4.1.3). `StandardScaler` is applied on top.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_q = StandardScaler()
scaler_m = StandardScaler()

X_quality_std = scaler_q.fit_transform(X_quality)
X_market_std  = scaler_m.fit_transform(X_market)

***
### 5.3. PCA Exploration

Before clustering we run PCA to understand how much variance can be captured in a small number of dimensions.  
Following the approach from Exercise 07 Task 2, we inspect the scree plot, then project the data to 2D for visualization.

In [ ]:
from src.clustering import fit_pca_with_scree

pca_q, var_ratio_q = fit_pca_with_scree(X_quality_std, title="Set A: quality profile")
pca_m, var_ratio_m = fit_pca_with_scree(X_market_std,  title="Set B: market segment")

In [ ]:
# cumulative variance explained by the first two PCs
print(f"Set A — first 2 PCs explain: {var_ratio_q[:2].sum():.1%}")
print(f"Set B — first 2 PCs explain: {var_ratio_m[:2].sum():.1%}")

# dominant loadings on PC1 for each set
market_cols = REVIEW_COLS + STRUCTURAL_COLS
print("\nSet A PC1 top loadings:")
for col, w in sorted(zip(REVIEW_COLS, pca_q.components_[0]), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {col:<35} {w:+.3f}")

print("\nSet B PC1 top loadings:")
for col, w in sorted(zip(market_cols, pca_m.components_[0]), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {col:<35} {w:+.3f}")

# project to 2D for later visualization
X_quality_pca2 = pca_q.transform(X_quality_std)[:, :2]
X_market_pca2  = pca_m.transform(X_market_std)[:, :2]

**Interpretation:**
- Set A: PC1 + PC2 explain **83.5%** of total variance — a reliable 2D projection. PC1 is a general quality axis: all 7 review dimensions load positively and nearly equally (+0.33 to +0.41), meaning PC1 captures overall guest satisfaction. PC2 (remaining ~16%) likely contrasts location against communication/checkin scores, since location has the weakest PC1 loading and would need its own axis to be distinguished.
- Set B: PC1 + PC2 explain **69.8%** of total variance — adequate but slightly lossier. PC1 again captures overall review quality (same positive loadings as Set A). PC2 is expected to capture the structural features — `price`, `accommodates`, and `amenities_count` all have near-zero loadings on PC1 (+0.051, −0.007, +0.094), so they contribute primarily through PC2, separating budget/compact listings from premium/spacious ones.
- Set A's 2D projection is trustworthy (83.5% > 60%). Set B's projection at 69.8% is acceptable but the structural features (price, size) are partially flattened — the true cluster separation in 10D space is better than the PCA scatter suggests.

***
### 5.4. K-Means Clustering

We use the elbow method (inertia) and silhouette score to choose `k`, then fit the final models and visualize in PCA 2D space.

#### 5.4.1. Choosing k via elbow and silhouette

In [ ]:
from src.clustering import sweep_kmeans
from src.plots    import plot_kmeans_sweep

sweep_q = sweep_kmeans(X_quality_std, k_range=range(2, 11), random_state=42)
sweep_m = sweep_kmeans(X_market_std,  k_range=range(2, 11), random_state=42)

plot_kmeans_sweep(sweep_q, sweep_m, labels=["Set A (quality)", "Set B (market)"])

**Decision:**

- Set A: elbow at k = 3, silhouette peak at k = 2 (0.81) → chosen **`k_quality = 3`**
- Set B: elbow at k = 4, silhouette flat after k = 2 (0.22–0.24) → chosen **`k_market = 4`**

**Why k = 3 for Set A despite the silhouette peak at k = 2?**

1. **Elbow criterion points to k = 3.** Inertia bends clearly at k = 3 — the standard primary criterion for K-Means model selection (Exercise 07, Task 1.2). Silhouette and elbow encode different qualities of a partition; we follow the convention of treating the elbow as the primary signal and silhouette as a sanity check.

2. **k = 2 would not add information beyond § 4.1.2.** A two-cluster split would essentially separate “high-rated mainstream” from “low-rated outliers” — two groups whose existence is already visible in the EDA rating boxplots (§ 4.1.2). The high silhouette of 0.81 reflects exactly this: the two groups sit at opposite ends of the rating scale, so the partition is internally compact, but it offers no segmentation beyond what EDA already shows. k = 3 instead reveals a mid-tier group (n = 331, median rating ≈ 4.33) that is not detectable from EDA distributions alone.

3. **The 2D projections support a three-cluster structure.** In the UMAP projection (§ 5.5, Set A), Cluster 1 (low-rated, n = 42) forms an isolated island, and Cluster 2 (mid-tier) occupies a distinct region to the right of the dominant top-rated mass. The three groups therefore correspond to genuine structure in feature space, not a forced over-segmentation.

We accept the lower silhouette (0.54, still indicating reasonable structure) in exchange for a finer, EDA-additive segmentation.

**Why k = 4 for Set B despite no clear silhouette signal?**

Silhouette is uninformative beyond k = 2 (all values 0.22–0.24), so the elbow at k = 4 is the decisive criterion. Four market segments — premium spacious, value mainstream, economy moderate-quality, and low-rated outliers — are also more business-meaningful than two undifferentiated halves.

In [ ]:
# set these after inspecting the sweep plots above
k_quality = 3
k_market  = 4

#### 5.4.2. Final model fit

In [ ]:
from src.clustering import fit_kmeans

km_q, labels_q = fit_kmeans(X_quality_std, k=k_quality, random_state=42)
km_m, labels_m = fit_kmeans(X_market_std,  k=k_market,  random_state=42)

print(f"Set A cluster sizes: { {i: int((labels_q==i).sum()) for i in range(k_quality)} }")
print(f"Set B cluster sizes: { {i: int((labels_m==i).sum()) for i in range(k_market)} }")

#### 5.4.3. Cluster visualization in PCA space

In [ ]:
from src.plots import plot_clusters_2d

plot_clusters_2d(
    X_quality_pca2, labels_q,
    X_market_pca2,  labels_m,
    titles=(f"Set A: quality profile (k={k_quality})",
            f"Set B: market segment (k={k_market})"),
    axis_labels=("PC1", "PC2"),
)

***
### 5.5. UMAP Comparison

We project both standardized feature sets to 2D with UMAP (Uniform Manifold Approximation and Projection), which preserves both local neighbourhood structure and global topology.

In [ ]:
print("Fitting UMAP for Set A...")
reducer_q = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_quality_umap = reducer_q.fit_transform(X_quality_std)

print("Fitting UMAP for Set B...")
reducer_m = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
X_market_umap  = reducer_m.fit_transform(X_market_std)

In [ ]:
plot_clusters_2d(
    X_quality_umap, labels_q,
    X_market_umap,  labels_m,
    titles=(f"Set A on UMAP (k={k_quality})",
            f"Set B on UMAP (k={k_market})"),
    axis_labels=("UMAP 1", "UMAP 2"),
)

**PCA vs UMAP comparison:**

Both methods reveal the same gross structure — one dominant cluster (Set A Cluster 0 / Set B Cluster 2) accounting for the majority of listings, with smaller outlier and premium sub-groups — but they emphasise different aspects:

| | PCA | UMAP |
|---|---|---|
| **Geometry** | Linear projection; clusters appear as overlapping ellipses along PC1 | Non-linear; preserves both local neighbourhood and approximate global structure |
| **Separation** | Cluster 1 (low-rated) at the negative PC1 extreme; Clusters 0 and 2 partially overlap | Set A: low-rated outliers form a fully isolated island; Clusters 0 and 2 are adjacent with a clear boundary. Set B: clusters are largely separated with only minor boundary overlap between Cluster 1 and 2 |
| **Outlier visibility** | Small clusters (n=42 / n=33) may be buried near the margin | Outlier clusters pulled into compact isolated blobs, immediately visible |
| **Speed** | Near-instant | Much faster than t-SNE; scales better to larger datasets |
| **Interpretability** | Axes carry meaning (PC1 ≈ overall quality, PC2 ≈ structural features) | Axes are arbitrary; relative distances within clusters are meaningful, distances between clusters are not directly comparable |

**Takeaway:** PCA is preferred for understanding *which features* drive separation; UMAP is preferred for *visually confirming* cluster compactness. Both agree that the low-rated outlier groups (Set A Cluster 1, Set B Cluster 3) are genuinely distinct isolated segments. The minor boundary overlap between Set B Cluster 1 and Cluster 2 in UMAP space is consistent with their similar review scores — the main differentiator between them is price and capacity rather than quality.

***
### 5.6. Cluster Profiles

We now translate each cluster label into an interpretable listing type.  
Median feature values per cluster (numeric) and top-3 category frequencies (categorical) are computed, then the review dimensions are visualized as a heatmap.

In [ ]:
from src.clustering import profile_clusters_numeric, profile_clusters_categorical

profile_q = profile_clusters_numeric(
    df=airbnb_df_norm.loc[idx_quality],
    labels=labels_q,
    cols=REVIEW_COLS + ["price", "accommodates", "amenities_count",
                        "host_is_superhost", "minimum_nights"],
)
display(profile_q)

In [ ]:
cat_profile_q = profile_clusters_categorical(
    df=airbnb_df_norm.loc[idx_quality],
    labels=labels_q,
    cols=["room_type", "neighbourhood_cleansed"],
    top_n=3,
)
display(cat_profile_q)

In [ ]:
profile_m = profile_clusters_numeric(
    df=airbnb_df_norm.loc[idx_market],
    labels=labels_m,
    cols=REVIEW_COLS + ["price", "accommodates", "amenities_count",
                        "host_is_superhost", "minimum_nights"],
)
display(profile_m)

cat_profile_m = profile_clusters_categorical(
    df=airbnb_df_norm.loc[idx_market],
    labels=labels_m,
    cols=["room_type", "neighbourhood_cleansed"],
    top_n=3,
)
display(cat_profile_m)

In [ ]:
from src.plots import plot_cluster_review_heatmap

plot_cluster_review_heatmap(
    profile_q, review_cols=REVIEW_COLS,
    title=f"Set A: review profile per cluster (k={k_quality})"
)
plot_cluster_review_heatmap(
    profile_m, review_cols=REVIEW_COLS,
    title=f"Set B: review profile per cluster (k={k_market})"
)

**Cluster names:**

*Set A — quality profile (k=3):*
- **Cluster 0 — "Top-rated stays"** (n=1 595, 81%): near-perfect scores across all 7 dimensions (rating 4.89, cleanliness 4.90, communication 4.95). Represents the mainstream high-quality offer in Zürich.
- **Cluster 1 — "Low-rated outliers"** (n=42, 2%): dramatically lower scores across the board (rating 2.50, value 2.50). A small but genuine low-quality tail. Concentrated in outer quartiere (Seebach 24%, Sihlfeld 12%, Hard 10%).
- **Cluster 2 — "Mid-tier quality"** (n=331, 17%): solid but not exceptional (rating 4.33, value 4.17). Likely newer hosts or listings receiving more mixed feedback. Disproportionately in Sihlfeld (12%) and Seebach (9%).

*Set B — market segment (k=4):*
- **Cluster 0 — "Economy moderate-quality"** (n=221, 11%): mid-range review scores (rating 4.10, value 4.00), modest price (CHF 118), small apartments (accommodates 2, amenities 26). Spread across Sihlfeld and Seebach.
- **Cluster 1 — "Premium spacious"** (n=579, 29%): highest price (CHF 213), largest capacity (accommodates 4, amenities 40), excellent ratings (4.88). Lakefront and upscale neighbourhoods: Seefeld (8%), Mühlebach (7%), Enge (6%).
- **Cluster 2 — "Value mainstream"** (n=1 135, 58%): lowest price (CHF 110), compact (accommodates 2), near-perfect ratings (4.86). The dominant Zürich offer. Spreads evenly across inner quartiere (Unterstrass, Sihlfeld, Langstrasse).
- **Cluster 3 — "Low-rated outliers"** (n=33, 2%): poor reviews (rating 2.33, value 2.33), average price (CHF 117). Outer quartiere (Seebach 27%, Hard 9%).

***
### 5.7. Geographic Distribution of Clusters

We project the cluster labels back onto the Zurich map, mirroring the setup from §4.1.3.  
This reveals whether clusters correspond to geographic zones (city centre vs. outskirts, lakefront vs. inland), or whether they cut across spatial boundaries.

In [ ]:
# attach cluster labels to the full normalized df (NaN for unlabelled rows)
airbnb_df_norm["cluster_quality"] = np.nan
airbnb_df_norm.loc[idx_quality, "cluster_quality"] = labels_q

airbnb_df_norm["cluster_market"] = np.nan
airbnb_df_norm.loc[idx_market, "cluster_market"] = labels_m

# build GeoDataFrame
airbnb_gdf_clust = gpd.GeoDataFrame(
    airbnb_df_norm,
    geometry=gpd.points_from_xy(
        airbnb_df_norm["longitude"], airbnb_df_norm["latitude"]
    ),
).set_crs(epsg=4326)

In [ ]:
# re-load geo layers (consistent with §4.1.3)
filepath_quart = "data/zurich_quartiere.gpkg"
filepath_for   = "data/forest_zurich.gpkg"
quartiere = gpd.read_file(filepath_quart, layer="stzh.adm_statistische_quartiere_map").to_crs(epsg=4326)
forest    = gpd.read_file(filepath_for).to_crs(epsg=4326)

In [ ]:
from src.plots import zurich_map_clusters

m_quality = zurich_map_clusters(
    airbnb_gdf_clust.dropna(subset=["cluster_quality"]),
    forest, quartiere,
    cluster_col="cluster_quality",
    title=f"Set A: quality clusters (k={k_quality})",
)
m_quality

In [ ]:
m_market = zurich_map_clusters(
    airbnb_gdf_clust.dropna(subset=["cluster_market"]),
    forest, quartiere,
    cluster_col="cluster_market",
    title=f"Set B: market clusters (k={k_market})",
)
m_market

In [ ]:
from src.plots import plot_cluster_share_per_quartier

plot_cluster_share_per_quartier(
    airbnb_df_norm.dropna(subset=["cluster_market"]),
    cluster_col="cluster_market",
    quartier_col="neighbourhood_cleansed",
    title=f"Set B: cluster share per Quartier (k={k_market})",
)

**Geographic observations:**

- **City centre** (Rathaus, Hochschulen, Lindenhof): dominated by Set B Cluster 2 ("Value mainstream") — small, affordable listings catering to short-stay business and leisure travellers. The central location compensates for the modest size and price.
- **Lake shore** (Seefeld, Mühlebach, Enge): strongly overrepresented by Set B Cluster 1 ("Premium spacious") — the Seefeld/Mühlebach/Enge belt accounts for 21% of Cluster 1 despite being only a fraction of total quartiere. This matches the §4.1.3 price hot-spot finding where lakefront areas command the highest prices.
- **Outer districts** (Seebach, Schwamendingen, Hard, Affoltern): overrepresented in both low-rated outlier clusters (Set A Cluster 1, Set B Cluster 3) and the mid-tier Set A Cluster 2. Outer-district listings are cheaper, smaller, and receive more mixed or poor feedback — consistent with the lower rental prices found in §4.3 for these areas.
- **Price hot-spot overlap**: the geographic footprint of Set B Cluster 1 ("Premium spacious") closely mirrors the §4.1.3 price heat-map — Seefeld, Mühlebach, Enge, and Hottingen are price peaks in both the EDA map and in Cluster 1's neighbourhood distribution, confirming that market segment is spatially determined by lakeside proximity and upscale residential character.

***
### 5.8. Discussion and Limitations

**Findings:**
- Set A identified **3 quality tiers** among reviewed listings: "Top-rated stays" (81%, rating ≈ 4.9), "Mid-tier quality" (17%, rating ≈ 4.3), and "Low-rated outliers" (2%, rating ≈ 2.5). The market is overwhelmingly high-quality — Zürich hosts maintain exceptionally strong review scores, with only a small tail of genuinely poor performers.
- Set B identified **4 market segments**: "Value mainstream" (58%, CHF 110, compact), "Premium spacious" (29%, CHF 213, 4 guests, lakeside), "Economy moderate-quality" (11%, CHF 118, mid scores), and "Low-rated outliers" (2%). Geographic distribution showed that the premium segment concentrates on the lake shore (Seefeld/Mühlebach/Enge) while the value mainstream spreads across the inner city — consistent with the price hot-spots identified in §4.1.3.

**Limitations:**
1. **Compressed score range**: as seen in §4.1.2, review scores are concentrated in the 4.5–5.0 band. This limits the discriminating power of Set A — small numeric differences drive the entire quality segmentation.
2. **K-Means assumes spherical, equally-sized clusters**: the real data may have elongated or unequal-density clusters. The UMAP visualization in §5.5 partially reveals this — the dominant cluster forms a dense core while outliers appear as isolated islands, and the boundary between Set B Cluster 1 and Cluster 2 shows minor overlap consistent with their shared review-score profile.
3. **Selection bias**: only listings with at least one review are included (24.1% of listings excluded). New or rarely-booked listings are excluded, so the clusters describe the *reviewed* market, not the full supply.
4. **No robustness check**: a single `random_state=42` and a single `dropna` strategy were used. Different seeds or imputation strategies would shift cluster boundaries.

### 6.0 Price prediction (MLR + Random Forest)...

### 7.0 Naive bayes

***
## 6. Naive Bayes Classification

### 6.1. Price Tier Definition

Rather than predicting the exact nightly price,
we frame this as a classification problem: given a listing's observable features, which
price tier does it belong to. Budget, mid-range, or premium?

Tiers are defined by the 33rd and 66th percentiles of the price distribution, giving three
roughly equal classes. Features used: `price` is excluded as the target is derived from it;
inputs are everything else

In [ ]:
from src.naive_bayes import plot_tier_distribution

fig, ax = plot_tier_distribution(airbnb_df_norm)
fig.savefig("images/naive_bayes_tier_distribution.png")

### 6.2. Model
We use a Mixed Naive Bayes model that applies the appropriate variant to each feature type:
- GaussianNB for continuous features (e.g. accommodates, amenities_count, review scores), which assumes features are normally distributed within each class
- BernoulliNB for binary features (amenity flags, bathroom and verification indicators)
- CategoricalNB for categorical features (room_type, neighbourhood_cleansed)

The likelihoods from all three sub-models are multiplied together (summed in log space), which is valid under the Naive Bayes independence assumption. The independence assumption is a known simplification. Features like accommodates and amenities_count are correlated, but using the correct distribution per feature type makes the model more accurate than applying GaussianNB to all features irregarding their type

In [ ]:
from src.naive_bayes import prepare_features, MixedNaiveBayes, plot_confusion_matrix, plot_price_by_predicted_tier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

test_size = 0.2

X_cont, X_bin, X_cat, y, prices, min_categories = prepare_features(airbnb_df_norm)

idx = np.arange(len(y))
idx_train, idx_test = train_test_split(
    idx, test_size=test_size, random_state=42,
    stratify=y.values
)

X_cont_train, X_cont_test = X_cont.iloc[idx_train], X_cont.iloc[idx_test]
X_bin_train,  X_bin_test  = X_bin.iloc[idx_train],  X_bin.iloc[idx_test]
X_cat_train,  X_cat_test  = X_cat.iloc[idx_train],  X_cat.iloc[idx_test]
y_train = y.iloc[idx_train]
y_test  = y.iloc[idx_test]
prices_test = prices.iloc[idx_test]

model = MixedNaiveBayes()
model.fit(X_cont_train, X_bin_train, X_cat_train, y_train, min_categories)
y_pred = model.predict(X_cont_test, X_bin_test, X_cat_test)



Below we can see the confusion matrix for the Naive Bayes classifier. As we can see, it performs somewhat better than the expected 33% baseline, however far better for budget and premium than for the mid-range options. Looking at the score, we see that budget and premium have the highest confidance but still miss a lot of themm (high recall). These cases mostly go to mid-range as the safest call, where we see high recall but low precision. 

In [ ]:
print(classification_report(y_test, y_pred, target_names=["budget", "mid-range", "premium"]))
fig, ax = plot_confusion_matrix(y_test, y_pred)
fig.savefig("images/naive_bayes_confusion.png")

Below is a plot of the price per predicited tier. From this we can see if the listings that were miscallsified are close in value to the actual class. Here we see that this is not really the case. While premium tier is mostly mid tier, for budget and mid-range, the predicitions are distributed wide into the respective other tiers. Therefore the model failure can not be purely attributed to the sharp cutoff

In [ ]:
fig, ax = plot_price_by_predicted_tier(prices_test, y_test, y_pred, airbnb_df_norm)
fig.savefig("images/naive_bayes_price_by_predicted.png")

In [ ]:
airbnb_df_norm